In [1]:
import matplotlib.pyplot as plt
import pickle

from pmbrl.model2 import Model
from pmbrl.data import Experiment_Data, get_data_expanded

import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
def find_params(m, inputs, target, initial_params, num_epochs=5000, learning_rate=.01):
    input_values = torch.tensor(inputs).reshape(1, len(inputs))
    param = torch.tensor(initial_params, requires_grad=True)
    target_value = torch.tensor(target).reshape(1, len(target))

    optimizer = optim.Adam([param], lr=learning_rate)
    criterion = nn.MSELoss(reduction='none')

    history = []

    for p in m.parameters():
        p.requires_grad = False

    for epoch in range(num_epochs):
        # Forward pass
        state_inputs = torch.concat([input_values, param.reshape(1, len(initial_params))], dim=1)
        output = m(state_inputs.float())  # Add batch dimension

        # Calculate the loss
        open_loss = criterion(output.float(), target_value.float())
        loss = torch.sqrt(open_loss.sum(axis=1).mean())
        

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        history.append((param.tolist(), output.tolist()[0], loss.item()))

        if (epoch + 1) % 100 == 0:
            # print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Input Param: {param.item():.4f}, Output: {output.item():.4f}')
            print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Input Param: {[round(p,4) for p in param.tolist()]}')
        return history


In [3]:
def optimize(row, model):
    m = model.transition_estimator.state_layer
    inputs = [row.s0, row.s1, row.s2, row.s3, row.a]
    targets = [row.s_0, row.s_1, row.s_2, row.s_3]
    init_param = [row.estimated_p0, row.estimated_p1]

    hist = find_params(m, inputs, targets, init_param)
    return hist[-1]


In [4]:
def predict(row, model):
    with torch.no_grad():
        m = model.transition_estimator.state_layer
        inputs = [row.s_0, row.s_1, row.s_2, row.s_3, row.a_]
        params = [row.new_estimated_p0, row.new_estimated_p1]
        # params = [row.estimated_p0, row.estimated_p1]

        input_values = torch.tensor(inputs).reshape(1, len(inputs))
        param = torch.tensor(params)

        for p in m.parameters():
            p.requires_grad = False

        state_inputs = torch.concat([input_values, param.reshape(1, len(params))], dim=1)
        output = m(state_inputs.float())  # Add batch dimension

    return [round(v, 3) for v in output.tolist()[0]]


In [5]:
rses = []
rses_norm = []

for m in range(10):
    print('model: ', m+1)
    nome_do_arquivo = f'triplet_{m+1}.pkl'

    with open(nome_do_arquivo, 'rb') as arquivo:
        exp = pickle.load(arquivo)
        data = exp['data']
        model = exp['model']

    del exp
    del arquivo

    data.evaluate_model(model, path='../testing_data.csv')
    results = data.get_evaluation_metrics()

    expansions = {
        'estimated_p': ['estimated_p0', 'estimated_p1'],
        's': ['s0', 's1', 's2', 's3'],
        's_': ['s_0', 's_1', 's_2', 's_3'],
        's__': ['s__0', 's__1', 's__2', 's__3'],
    }

    df = data.evaluation_data.copy()
    df = get_data_expanded(df, expansions)

    df[['new_estimated_p', 'new_estimated_s', 'param_rse']] = df.apply(lambda row: optimize(row,model), axis=1, result_type='expand')
    
    expansions = {
        'new_estimated_p': ['new_estimated_p0', 'new_estimated_p1'],
    }

    final_df = data.evaluation_data.copy()
    final_df = get_data_expanded(df, expansions)

    final_df['new_estimated_s'] = final_df.apply(lambda row: predict(row,model), axis=1)

    df_metrics = final_df.copy()
    df_metrics['estimated_s'] = df_metrics['new_estimated_s']
    results = data.get_evaluation_metrics(df_metrics)

    rses.append(results.rse.mean())
    rses_norm.append(results.rse_normalized.mean())

    del df
    del results
    del df_metrics
    del final_df
    del data
    del model

model:  1
model:  2
model:  3
model:  4
model:  5
model:  6
model:  7
model:  8
model:  9
model:  10


In [6]:
print('rse:', sum(rses)/len(rses))
print('rses_norm:', sum(rses_norm)/len(rses_norm))

rse: 0.05143187588152327
rses_norm: 0.5083357592771655
